# NLP - Notebook 1 : Corpus et Index FAISS
## Systeme RAG pour Documents de Maintenance Turbofan

Ce notebook construit la base documentaire du systeme RAG
a partir de 4 sources officielles et gratuites :

- Wikipedia : articles techniques sur turbofan et maintenance
- FAA : documents officiels de maintenance aeronautique
- SKYbrary : base de connaissance EUROCONTROL/EASA
- EASA : bulletins de securite europeens

Pipeline :
1. Collecte du texte depuis les 4 sources
2. Nettoyage et normalisation
3. Chunking avec overlap
4. Generation des embeddings avec Sentence-BERT
5. Construction et sauvegarde de l index FAISS

Environnement : Kaggle avec GPU T4.

## Cellule 1 - Installation

wikipedia-api : acces propre a l API Wikipedia
beautifulsoup4 : scraping SKYbrary et EASA
sentence-transformers : generation des embeddings
faiss-cpu : construction de l index de similarite
PyMuPDF : lecture des PDFs FAA

In [1]:
!pip install wikipedia-api beautifulsoup4 sentence-transformers faiss-cpu PyMuPDF requests nltk tqdm -q

print('Installation terminee.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.4/108.4 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 69.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
Installation terminee.


## Cellule 2 - Imports et configuration

Tous les chemins et parametres sont definis ici.
Si tu veux modifier quelque chose, c est dans cette cellule.

In [2]:
import os
import re
import json
import time
import random
import requests
import numpy as np
import fitz
import faiss
import nltk
import wikipediaapi
from bs4 import BeautifulSoup
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')
else:
    print('CPU utilise.')

# Chemins de travail
BASE        = '/kaggle/working/rag_system/'
PDFS_DIR    = BASE + 'pdfs/'
INDEX_DIR   = BASE + 'index/'
RESULTS_DIR = BASE + 'results/'

for p in [PDFS_DIR, INDEX_DIR, RESULTS_DIR]:
    os.makedirs(p, exist_ok=True)

# Parametres du chunking
CHUNK_SIZE     = 400  # mots par chunk
OVERLAP_TOKENS = 50   # mots d overlap entre chunks

print(f'Dossier de travail : {BASE}')
print('Configuration terminee.')

GPU : Tesla T4
Dossier de travail : /kaggle/working/rag_system/
Configuration terminee.


## Cellule 3 - Source 1 : Wikipedia

On telecharge le contenu complet de 50 articles Wikipedia
directement lies a la maintenance des moteurs turbofan.

Wikipedia couvre tres bien :
- Les composants du moteur et leur role
- Les modes de defaillance courants
- Les procedures d inspection generales
- Les standards et certifications

L API Wikipedia retourne le texte complet de chaque article
sans HTML ni publicites, directement utilisable.

In [3]:
# Articles Wikipedia pertinents pour la maintenance turbofan
WIKIPEDIA_ARTICLES = [
    # Moteurs specifiques
    'CFM56', 'CFM International LEAP', 'General Electric GE90',
    'Pratt & Whitney PW4000', 'Rolls-Royce Trent', 'IAE V2500',
    'General Electric GEnx', 'Pratt & Whitney PW1000G',

    # Technologie turbofan
    'Turbofan', 'Turbojet', 'Turboshaft', 'Jet engine',
    'Aircraft engine', 'Gas turbine', 'Bypass ratio',

    # Composants critiques
    'Compressor (gas turbine)', 'Turbine blade', 'Combustion chamber',
    'Turbine', 'Fan (aeronautics)', 'Afterburner',
    'Thrust reverser', 'Nozzle',

    # Modes de defaillance
    'Compressor stall', 'Foreign object damage',
    'Hot section', 'Turbine blade cooling',
    'Metal fatigue', 'Creep (deformation)',
    'Corrosion', 'Erosion',

    # Maintenance et inspection
    'Aircraft maintenance', 'Borescope',
    'Airworthiness directive', 'Engine overhaul',
    'Non-destructive testing', 'Ultrasonic testing',
    'Magnetic particle inspection', 'Dye penetrant inspection',
    'Aircraft engine starting', 'Engine washing',

    # Standards et reglements
    'Federal Aviation Administration',
    'European Union Aviation Safety Agency',
    'ICAO', 'Airworthiness',
    'Aviation safety', 'Aircraft certification',
    'Continued airworthiness',

    # Maintenance predictive
    'Predictive maintenance', 'Condition monitoring',
    'Health and usage monitoring systems',
    'Prognostics and health management',
    'Remaining useful life'
]

print(f'Articles Wikipedia a telecharger : {len(WIKIPEDIA_ARTICLES)}')

# Initialiser l API Wikipedia
wiki = wikipediaapi.Wikipedia(
    user_agent='TurbofanRAG/1.0 (research project)',
    language='en'
)

wikipedia_docs = []
failed_wiki    = []

for article_name in tqdm(WIKIPEDIA_ARTICLES, desc='Wikipedia'):
    page = wiki.page(article_name)

    if page.exists() and len(page.text) > 500:
        wikipedia_docs.append({
            'text'   : page.text,
            'source' : f'Wikipedia : {article_name}',
            'title'  : page.title,
            'url'    : page.fullurl
        })
    else:
        failed_wiki.append(article_name)

    time.sleep(0.5)  # respecter l API Wikipedia

print(f'\nArticles telecharges : {len(wikipedia_docs)}')
print(f'Echecs               : {len(failed_wiki)}')
if failed_wiki:
    print(f'Articles non trouves : {failed_wiki}')

total_words = sum(len(d['text'].split()) for d in wikipedia_docs)
print(f'Mots total Wikipedia : {total_words:,}')

# Afficher un exemple
if wikipedia_docs:
    ex = wikipedia_docs[0]
    print(f'\nExemple - {ex["title"]} :')
    print(ex['text'][:400])

Articles Wikipedia a telecharger : 53


Wikipedia: 100%|██████████| 53/53 [00:34<00:00,  1.55it/s]


Articles telecharges : 46
Echecs               : 7
Articles non trouves : ['Compressor (gas turbine)', 'Fan (aeronautics)', 'Hot section', 'Turbine blade cooling', 'Engine washing', 'Aircraft certification', 'Continued airworthiness']
Mots total Wikipedia : 143,384

Exemple - CFM International CFM56 :
The CFM International CFM56 (U.S. military designation F108) series is a Franco-American family of high-bypass turbofan aircraft engines made by CFM International (CFMI), with a thrust range of 18,500 to 34,000 lbf (82 to 150 kN). CFMI is a 50–50 joint-owned company of Safran Aircraft Engines (formerly known as Snecma) of France, and GE Aerospace (GE) of the United States. GE produces the high-pre


## Cellule 4 - Source 2 : Documents FAA

On telecharge 4 documents officiels de la FAA
directement lies a la maintenance des moteurs.

Ces documents contiennent des procedures officielles,
des intervalles d inspection certifies, et des
criteres de remplacement reconnus par les autorites.

In [4]:
def download_pdf(url, save_path, timeout=30):
    try:
        headers  = {'User-Agent': 'Mozilla/5.0 (research purpose)'}
        response = requests.get(url, headers=headers, timeout=timeout)
        if response.status_code == 200 and len(response.content) > 1000:
            with open(save_path, 'wb') as f:
                f.write(response.content)
            return True
        return False
    except Exception:
        return False


def extract_text_from_pdf(pdf_path):
    try:
        doc   = fitz.open(pdf_path)
        text  = ''
        for page in doc:
            text += page.get_text() + '\n'
        doc.close()
        lines = [l.strip() for l in text.split('\n') if len(l.strip()) > 30]
        text  = ' '.join(lines)
        text  = re.sub(r'\s+', ' ', text).strip()
        text  = re.sub(r'[^\x00-\x7F]+', ' ', text)
        return text if len(text) > 500 else None
    except Exception:
        return None


faa_documents = [
    {
        'url'      : 'https://www.faa.gov/documentlibrary/media/advisory_circular/ac_43.13-1b_w-chg1.pdf',
        'filename' : 'faa_ac43_inspection_repair.pdf',
        'title'    : 'FAA AC43.13-1B - Aircraft Inspection and Repair Methods'
    },
    {
        'url'      : 'https://www.faa.gov/documentLibrary/media/Advisory_Circular/AC_120-113.pdf',
        'filename' : 'faa_ac120_engine_interval.pdf',
        'title'    : 'FAA AC120-113 - Engine Time In Service Interval Extensions'
    },
    {
        'url'      : 'https://www.faa.gov/documentLibrary/media/Advisory_Circular/AC_20-147A.pdf',
        'filename' : 'faa_ac20_turbofan_icing.pdf',
        'title'    : 'FAA AC20-147A - Turbofan Engine Induction System Icing'
    },
    {
        'url'      : 'https://www.faa.gov/documentLibrary/media/Advisory_Circular/AC_25.933-1.pdf',
        'filename' : 'faa_ac25_thrust_reverser.pdf',
        'title'    : 'FAA AC25.933 - Turbofan Thrust Reverser Maintenance'
    }
]

print('Telechargement des documents FAA...')
faa_docs = []

for doc in faa_documents:
    save_path = PDFS_DIR + doc['filename']
    print(f'  Telechargement : {doc["filename"]}...')

    success = download_pdf(doc['url'], save_path)

    if success:
        text = extract_text_from_pdf(save_path)
        if text:
            size = os.path.getsize(save_path) / 1e6
            faa_docs.append({
                'text'   : text,
                'source' : f'FAA : {doc["title"]}',
                'title'  : doc['title'],
                'url'    : doc['url']
            })
            print(f'  OK : {doc["filename"]} ({size:.1f} MB, {len(text.split()):,} mots)')
        else:
            print(f'  Texte non extractible : {doc["filename"]}')
    else:
        print(f'  Echec telechargement : {doc["filename"]}')

    time.sleep(1)

print(f'\nDocuments FAA charges : {len(faa_docs)}')

Telechargement des documents FAA...
  Telechargement : faa_ac43_inspection_repair.pdf...
  OK : faa_ac43_inspection_repair.pdf (21.1 MB, 182,932 mots)
  Telechargement : faa_ac120_engine_interval.pdf...
  OK : faa_ac120_engine_interval.pdf (0.2 MB, 3,045 mots)
  Telechargement : faa_ac20_turbofan_icing.pdf...
  OK : faa_ac20_turbofan_icing.pdf (0.7 MB, 22,513 mots)
  Telechargement : faa_ac25_thrust_reverser.pdf...
  OK : faa_ac25_thrust_reverser.pdf (4.6 MB, 9,766 mots)

Documents FAA charges : 4


## Cellule 5 - Source 3 : SKYbrary

SKYbrary est une base de connaissance aviation creee par
EUROCONTROL et l EASA. Elle contient des articles detailles
sur la maintenance, la securite, et les procedures aeronautiques.

On scrape le contenu textuel de chaque article
en supprimant les elements HTML inutiles.

In [5]:
SKYBRARY_ARTICLES = [
    'https://skybrary.aero/articles/turbofan-engine',
    'https://skybrary.aero/articles/engine-failure-after-v1',
    'https://skybrary.aero/articles/jet-engine-bird-strike',
    'https://skybrary.aero/articles/engine-fire',
    'https://skybrary.aero/articles/compressor-stall',
    'https://skybrary.aero/articles/foreign-object-debris-fod',
    'https://skybrary.aero/articles/aircraft-maintenance',
    'https://skybrary.aero/articles/airworthiness',
    'https://skybrary.aero/articles/continuing-airworthiness',
    'https://skybrary.aero/articles/maintenance-error',
    'https://skybrary.aero/articles/line-maintenance',
    'https://skybrary.aero/articles/base-maintenance',
    'https://skybrary.aero/articles/engine-oil-system',
    'https://skybrary.aero/articles/borescope-inspection',
    'https://skybrary.aero/articles/airworthiness-directive'
]


def scrape_skybrary(url):
    """
    Recupere le contenu textuel d un article SKYbrary.
    Supprime les menus, publicites, et elements de navigation.
    Garde uniquement le contenu principal de l article.
    """
    try:
        headers  = {'User-Agent': 'Mozilla/5.0 (research purpose)'}
        response = requests.get(url, headers=headers, timeout=15)

        if response.status_code != 200:
            return None, None

        soup = BeautifulSoup(response.text, 'html.parser')

        # Supprimer les elements inutiles
        for tag in soup(['script', 'style', 'nav', 'header',
                         'footer', 'aside', 'form']):
            tag.decompose()

        # Recuperer le titre
        title = soup.find('h1')
        title = title.get_text().strip() if title else url.split('/')[-1]

        # Recuperer le contenu principal
        content = soup.find('main') or soup.find('article') or soup.find('body')
        if not content:
            return None, None

        text = content.get_text(separator=' ')
        text = re.sub(r'\s+', ' ', text).strip()
        text = re.sub(r'[^\x00-\x7F]+', ' ', text)

        return title, text if len(text) > 200 else None

    except Exception:
        return None, None


print('Scraping SKYbrary...')
skybrary_docs = []
failed_sky    = []

for url in tqdm(SKYBRARY_ARTICLES, desc='SKYbrary'):
    title, text = scrape_skybrary(url)

    if text:
        skybrary_docs.append({
            'text'   : text,
            'source' : f'SKYbrary : {title}',
            'title'  : title,
            'url'    : url
        })
    else:
        failed_sky.append(url)

    time.sleep(1)

print(f'\nArticles SKYbrary charges : {len(skybrary_docs)}')
print(f'Echecs                    : {len(failed_sky)}')

if skybrary_docs:
    ex = skybrary_docs[0]
    print(f'\nExemple - {ex["title"]} :')
    print(ex['text'][:300])

Scraping SKYbrary...


SKYbrary: 100%|██████████| 15/15 [00:25<00:00,  1.69s/it]


Articles SKYbrary charges : 10
Echecs                    : 5

Exemple - Turbofan Engine :
Skip to main content You are here Home   Portals   Enhancing Safety   Flight Technical Turbofan Engine Turbofan Engine Article Information Category: Flight Technical Content source: SKYbrary Content control: SKYbrary Description A turbofan engine, sometimes referred to as a fanjet or bypass engine, 


## Cellule 6 - Source 4 : EASA

L EASA (European Union Aviation Safety Agency) publie
des Safety Information Bulletins et des documents techniques
sur la maintenance des moteurs aeronautiques.

Ces documents sont l equivalent europeen des ADs FAA.

In [6]:
EASA_URLS = [
    {
        'url'   : 'https://www.easa.europa.eu/en/document-library/easy-access-rules/online-publications/easy-access-rules-continuing-airworthiness',
        'title' : 'EASA Continuing Airworthiness Rules'
    },
    {
        'url'   : 'https://www.easa.europa.eu/en/domains/aircraft-products/aircraft-maintenance',
        'title' : 'EASA Aircraft Maintenance Guidelines'
    },
    {
        'url'   : 'https://www.easa.europa.eu/en/domains/aircraft-products/engines',
        'title' : 'EASA Engine Certification and Maintenance'
    }
]


def scrape_easa(url, title):
    """
    Recupere le contenu textuel d une page EASA.
    """
    try:
        headers  = {'User-Agent': 'Mozilla/5.0 (research purpose)'}
        response = requests.get(url, headers=headers, timeout=15)

        if response.status_code != 200:
            return None

        soup = BeautifulSoup(response.text, 'html.parser')

        for tag in soup(['script', 'style', 'nav', 'header', 'footer', 'aside']):
            tag.decompose()

        content = soup.find('main') or soup.find('article') or soup.find('body')
        if not content:
            return None

        text = content.get_text(separator=' ')
        text = re.sub(r'\s+', ' ', text).strip()
        text = re.sub(r'[^\x00-\x7F]+', ' ', text)

        return text if len(text) > 200 else None

    except Exception:
        return None


print('Scraping EASA...')
easa_docs  = []
failed_easa = []

for doc in tqdm(EASA_URLS, desc='EASA'):
    text = scrape_easa(doc['url'], doc['title'])

    if text:
        easa_docs.append({
            'text'   : text,
            'source' : f'EASA : {doc["title"]}',
            'title'  : doc['title'],
            'url'    : doc['url']
        })
        print(f'  OK : {doc["title"]} ({len(text.split()):,} mots)')
    else:
        failed_easa.append(doc['url'])
        print(f'  Echec : {doc["title"]}')

    time.sleep(1)

print(f'\nDocuments EASA charges : {len(easa_docs)}')

Scraping EASA...


EASA:   0%|          | 0/3 [00:00<?, ?it/s]

  OK : EASA Continuing Airworthiness Rules (410 mots)


EASA:  33%|███▎      | 1/3 [00:01<00:02,  1.20s/it]

  Echec : EASA Aircraft Maintenance Guidelines


EASA:  67%|██████▋   | 2/3 [00:02<00:01,  1.09s/it]

  Echec : EASA Engine Certification and Maintenance


EASA: 100%|██████████| 3/3 [00:03<00:00,  1.08s/it]


Documents EASA charges : 1


## Cellule 7 - Assemblage du corpus complet

On combine toutes les sources en un seul corpus.
On affiche les statistiques pour verifier que tout est bien charge.

In [7]:
# Combiner toutes les sources
all_docs = wikipedia_docs + faa_docs + skybrary_docs + easa_docs

print('='*55)
print('CORPUS COMPLET')
print('='*55)
print(f'Wikipedia  : {len(wikipedia_docs):>5} documents')
print(f'FAA        : {len(faa_docs):>5} documents')
print(f'SKYbrary   : {len(skybrary_docs):>5} documents')
print(f'EASA       : {len(easa_docs):>5} documents')
print('-'*55)
print(f'TOTAL      : {len(all_docs):>5} documents')

total_words = sum(len(d['text'].split()) for d in all_docs)
avg_words   = total_words / len(all_docs) if all_docs else 0
print(f'\nMots total   : {total_words:,}')
print(f'Mots moyenne : {avg_words:.0f} par document')

# Sauvegarder le corpus brut
corpus_path = INDEX_DIR + 'corpus.json'
corpus_save = [{'text': d['text'], 'source': d['source'],
                'title': d['title'], 'url': d['url']}
               for d in all_docs]
with open(corpus_path, 'w', encoding='utf-8') as f:
    json.dump(corpus_save, f, ensure_ascii=False, indent=2)
print(f'\nCorpus sauvegarde : {corpus_path}')

CORPUS COMPLET
Wikipedia  :    46 documents
FAA        :     4 documents
SKYbrary   :    10 documents
EASA       :     1 documents
-------------------------------------------------------
TOTAL      :    61 documents

Mots total   : 372,287
Mots moyenne : 6103 par document

Corpus sauvegarde : /kaggle/working/rag_system/index/corpus.json


## Cellule 8 - Nettoyage du texte

On applique un nettoyage uniforme sur tous les documents
quelque soit leur source.

On supprime les artefacts de scraping comme les menus
repetes, les boutons, les liens, et les espaces excessifs.

In [8]:
def clean_text(text):
    """
    Nettoyage uniforme applicable a toutes les sources.
    """
    # Supprimer les URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Supprimer les caracteres speciaux excessifs
    text = re.sub(r'[^a-zA-Z0-9\s\.\,\;\:\-\(\)\[\]\/\°\%]', ' ', text)

    # Supprimer les lignes trop courtes (boutons, menus)
    lines = text.split('.')
    lines = [l.strip() for l in lines if len(l.strip()) > 20]
    text  = '. '.join(lines)

    # Normaliser les espaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text


print('Nettoyage du texte...')
cleaned_docs = []

for doc in tqdm(all_docs, desc='Nettoyage'):
    cleaned = clean_text(doc['text'])
    if len(cleaned.split()) > 100:  # garder seulement les docs substantiels
        cleaned_docs.append({
            'text'   : cleaned,
            'source' : doc['source'],
            'title'  : doc['title'],
            'url'    : doc['url']
        })

print(f'Documents apres nettoyage : {len(cleaned_docs)}/{len(all_docs)}')

# Exemple avant / apres nettoyage
print('\nAvant nettoyage :')
print(all_docs[0]['text'][:200])
print('\nApres nettoyage :')
print(cleaned_docs[0]['text'][:200])

Nettoyage du texte...


Nettoyage: 100%|██████████| 61/61 [00:00<00:00, 285.63it/s]

Documents apres nettoyage : 61/61

Avant nettoyage :
The CFM International CFM56 (U.S. military designation F108) series is a Franco-American family of high-bypass turbofan aircraft engines made by CFM International (CFMI), with a thrust range of 18,500

Apres nettoyage :
The CFM International CFM56 (U. military designation F108) series is a Franco-American family of high-bypass turbofan aircraft engines made by CFM International (CFMI), with a thrust range of 18,500 t


## Cellule 9 - Chunking avec overlap

On decoupe chaque document en chunks de 400 mots maximum
avec un overlap de 50 mots entre chunks consecutifs.

On respecte les frontieres de phrases avec sent_tokenize.
On ne coupe jamais une phrase en plein milieu.

Chaque chunk garde les metadonnees de son document source
pour pouvoir citer la source dans la reponse finale.

In [9]:
def chunk_document(text, source, title, url,
                   chunk_size=CHUNK_SIZE, overlap=OVERLAP_TOKENS):
    """
    Decoupe un document en chunks de max chunk_size mots.
    Respecte les frontieres de phrases.
    Ajoute un overlap entre chunks consecutifs.
    """
    sentences = sent_tokenize(text)
    if not sentences:
        return []

    chunks        = []
    current_sents = []
    current_count = 0
    chunk_id      = 0

    for sent in sentences:
        sent_words = len(sent.split())

        if current_count + sent_words > chunk_size and current_sents:
            chunks.append({
                'text'     : ' '.join(current_sents),
                'source'   : source,
                'title'    : title,
                'url'      : url,
                'chunk_id' : chunk_id
            })
            chunk_id += 1

            # Overlap
            overlap_sents = []
            overlap_count = 0
            for s in reversed(current_sents):
                s_words = len(s.split())
                if overlap_count + s_words <= overlap:
                    overlap_sents.insert(0, s)
                    overlap_count += s_words
                else:
                    break

            current_sents = overlap_sents
            current_count = overlap_count

        current_sents.append(sent)
        current_count += sent_words

    if current_sents:
        chunks.append({
            'text'     : ' '.join(current_sents),
            'source'   : source,
            'title'    : title,
            'url'      : url,
            'chunk_id' : chunk_id
        })

    return chunks


print('Chunking en cours...')
all_chunks = []

for doc in tqdm(cleaned_docs, desc='Chunking'):
    chunks = chunk_document(
        text   = doc['text'],
        source = doc['source'],
        title  = doc['title'],
        url    = doc['url']
    )
    all_chunks.extend(chunks)

print(f'\nChunks generes : {len(all_chunks)}')
print(f'Moyenne        : {len(all_chunks)/len(cleaned_docs):.1f} chunks par document')

chunk_lengths = [len(c['text'].split()) for c in all_chunks]
print(f'Mots par chunk - Moyenne : {np.mean(chunk_lengths):.0f}')
print(f'Mots par chunk - Max     : {np.max(chunk_lengths)}')
print(f'Mots par chunk - Min     : {np.min(chunk_lengths)}')

# Repartition par source
print('\nRepartition des chunks par source :')
source_counts = {}
for chunk in all_chunks:
    src = chunk['source'].split(':')[0].strip()
    source_counts[src] = source_counts.get(src, 0) + 1
for src, count in source_counts.items():
    print(f'  {src:<12} : {count} chunks')

# Sauvegarder les chunks
chunks_path = INDEX_DIR + 'chunks_metadata.json'
with open(chunks_path, 'w', encoding='utf-8') as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)
print(f'\nChunks sauvegardes : {chunks_path}')

Chunking en cours...


Chunking: 100%|██████████| 61/61 [00:00<00:00, 168.25it/s]


Chunks generes : 1081
Moyenne        : 17.7 chunks par document
Mots par chunk - Moyenne : 374
Mots par chunk - Max     : 486
Mots par chunk - Min     : 54

Repartition des chunks par source :
  Wikipedia    : 431 chunks
  FAA          : 614 chunks
  SKYbrary     : 34 chunks
  EASA         : 2 chunks

Chunks sauvegardes : /kaggle/working/rag_system/index/chunks_metadata.json


## Cellule 10 - Generation des embeddings avec Sentence-BERT

On convertit chaque chunk en vecteur de 384 dimensions.

Le modele all-MiniLM-L6-v2 est leger (22M params) mais
tres performant pour la similarite semantique.
Deux chunks qui parlent de la meme chose auront des
vecteurs proches meme s ils utilisent des mots differents.

In [10]:
print('Chargement Sentence-BERT...')
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print('Modele charge.')

chunk_texts = [c['text'] for c in all_chunks]

print(f'Generation des embeddings pour {len(chunk_texts)} chunks...')

embeddings = embedder.encode(
    chunk_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(f'\nEmbeddings generes.')
print(f'  Forme    : {embeddings.shape}')
print(f'  Chunks   : {embeddings.shape[0]}')
print(f'  Dims     : {embeddings.shape[1]}')

embeddings_path = INDEX_DIR + 'embeddings.npy'
np.save(embeddings_path, embeddings)
size_mb = os.path.getsize(embeddings_path) / 1e6
print(f'  Sauvegarde : {embeddings_path} ({size_mb:.1f} MB)')

Chargement Sentence-BERT...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modele charge.
Generation des embeddings pour 1081 chunks...


Batches:   0%|          | 0/17 [00:00<?, ?it/s]


Embeddings generes.
  Forme    : (1081, 384)
  Chunks   : 1081
  Dims     : 384
  Sauvegarde : /kaggle/working/rag_system/index/embeddings.npy (1.7 MB)


## Cellule 11 - Construction de l index FAISS

On construit l index FAISS et on le teste avec
des questions reelles qu un ingenieur de maintenance poserait.

IndexFlatIP avec vecteurs normalises = cosine similarity exacte.
Score proche de 1.0 = tres similaire.
Score proche de 0.0 = pas similaire.

In [11]:
print('Construction de l index FAISS...')

dimension = embeddings.shape[1]
index     = faiss.IndexFlatIP(dimension)
index.add(embeddings.astype(np.float32))

print(f'Index construit.')
print(f'  Vecteurs indexes : {index.ntotal}')
print(f'  Dimension        : {dimension}')

index_path = INDEX_DIR + 'faiss_index.index'
faiss.write_index(index, index_path)
size_mb = os.path.getsize(index_path) / 1e6
print(f'  Sauvegarde       : {index_path} ({size_mb:.1f} MB)')

# Test avec des vraies questions d un ingenieur de maintenance
test_queries = [
    'what is the inspection interval for turbofan compressor blades',
    'how to detect compressor stall in jet engine',
    'oil filter replacement procedure aircraft engine',
    'what causes turbine blade erosion',
    'airworthiness directive engine maintenance requirements',
    'borescope inspection procedure turbofan'
]

print('\nTest de l index avec des questions de maintenance :')
print('='*65)

for query in test_queries:
    query_emb = embedder.encode(
        [query], normalize_embeddings=True
    ).astype(np.float32)

    scores, indices = index.search(query_emb, k=3)

    print(f'\nQuestion : {query}')
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0])):
        chunk = all_chunks[idx]
        print(f'  Resultat {rank+1} (score {score:.3f}) :')
        print(f'    Source : {chunk["source"]}')
        print(f'    Texte  : {chunk["text"][:120]}...')

Construction de l index FAISS...
Index construit.
  Vecteurs indexes : 1081
  Dimension        : 384
  Sauvegarde       : /kaggle/working/rag_system/index/faiss_index.index (1.7 MB)

Test de l index avec des questions de maintenance :

Question : what is the inspection interval for turbofan compressor blades
  Resultat 1 (score 0.572) :
    Source : FAA : FAA AC120-113 - Engine Time In Service Interval Extensions
    Texte  : 411(a)(2) of this chapter. (2) An approved aircraft inspection program approved under 135. 419 of this chapter and curre...
  Resultat 2 (score 0.476) :
    Source : FAA : FAA AC43.13-1B - Aircraft Inspection and Repair Methods
    Texte  : Steel Blade Inspection. The inspection of steel blades may be accomplished by either visual, fluorescent penetrant (see ...
  Resultat 3 (score 0.473) :
    Source : SKYbrary : Continuing Airworthiness
    Texte  : Inspection methods and intervals, repair actions, modifications, and timescales are all part of Continuing Airwort

## Cellule 12 - Bilan final

Resume complet de tout ce qui a ete construit.
Ces fichiers seront charges dans le Notebook 2
pour le pipeline RAG avec LLaMA.

In [12]:
print('='*60)
print('BILAN - Notebook 1 : Corpus et Index FAISS')
print('='*60)

print('\nCorpus :')
print(f'  Wikipedia  : {len(wikipedia_docs)} articles')
print(f'  FAA        : {len(faa_docs)} documents')
print(f'  SKYbrary   : {len(skybrary_docs)} articles')
print(f'  EASA       : {len(easa_docs)} pages')
print(f'  Total docs : {len(cleaned_docs)}')
print(f'  Total chunks : {len(all_chunks)}')

print('\nFichiers generes :')
for root, dirs, files in os.walk(BASE):
    for f in files:
        full = os.path.join(root, f)
        size = os.path.getsize(full) / 1e6
        print(f'  {full}  ({size:.1f} MB)')

print('\nFichiers pour le Notebook 2 :')
print(f'  {INDEX_DIR}faiss_index.index     <- index de recherche')
print(f'  {INDEX_DIR}chunks_metadata.json  <- textes et sources')
print(f'  {INDEX_DIR}embeddings.npy        <- vecteurs')

print('\nProchaine etape - Notebook 2 : RAG + LLaMA Pipeline')
print('  Charger l index FAISS')
print('  Charger LLaMA-3.2-3b-Instruct')
print('  Pipeline RAG complet')
print('  Mode summarisation')
print('  Mode Q&A')
print('  Evaluation RAGAS')

print('\n' + '='*60)
print('Notebook 1 termine.')
print('='*60)

BILAN - Notebook 1 : Corpus et Index FAISS

Corpus :
  Wikipedia  : 46 articles
  FAA        : 4 documents
  SKYbrary   : 10 articles
  EASA       : 1 pages
  Total docs : 61
  Total chunks : 1081

Fichiers generes :
  /kaggle/working/rag_system/pdfs/faa_ac25_thrust_reverser.pdf  (4.6 MB)
  /kaggle/working/rag_system/pdfs/faa_ac120_engine_interval.pdf  (0.2 MB)
  /kaggle/working/rag_system/pdfs/faa_ac43_inspection_repair.pdf  (21.1 MB)
  /kaggle/working/rag_system/pdfs/faa_ac20_turbofan_icing.pdf  (0.7 MB)
  /kaggle/working/rag_system/index/corpus.json  (2.5 MB)
  /kaggle/working/rag_system/index/chunks_metadata.json  (2.8 MB)
  /kaggle/working/rag_system/index/embeddings.npy  (1.7 MB)
  /kaggle/working/rag_system/index/faiss_index.index  (1.7 MB)

Fichiers pour le Notebook 2 :
  /kaggle/working/rag_system/index/faiss_index.index     <- index de recherche
  /kaggle/working/rag_system/index/chunks_metadata.json  <- textes et sources
  /kaggle/working/rag_system/index/embeddings.npy     